<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/05-loss-optimization-training-dynamics.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **损失函数、优化与训练动态** {#loss-optimization-training-dynamics}

第 04 章解释了反向模式自动微分如何把一个标量目标转化为梯度。训练还剩下两个关键决定：**什么样的标量目标才代表有用的行为**，以及**这些梯度应如何在大量带噪步骤中改变参数**。损失函数规定学习信号，优化器规定更新动态。二者彼此耦合：即使优化器实现完全正确，也可能在优化不合适的目标；即使损失设计良好，也可能因不稳定的学习率、较差的初始化或不匹配的 batch 方案而训练失败。

本章沿着一次训练更新，从目标设计一直走到诊断。它区分数据拟合与正则化，推导 SGD、动量、自适应方法和 AdamW 背后的更新规则，并把这些方程连接到实际 PyTorch 训练中真正重要的选择：学习率调度、warmup、梯度累积、裁剪、混合精度、初始化和测量。

### **目标函数、经验风险与学习信号** {#objectives-empirical-risk-learning-signals}

对于数据集 $\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{N}$，监督训练通常最小化一个带正则的经验目标：

$$
J(\theta)
=
\underbrace{\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i)}_{\text{data fit / empirical risk}}
+
\underbrace{\lambda\,\Omega(\theta)}_{\text{regularization or prior preference}}.
$$

损失 $\ell$ 将一个预测-目标对转化为标量差异。数据集平均值是在已观测数据分布上对期望风险的估计。正则项表达的是单个样本无法提供的偏好，例如更小的权重、更平滑的函数、稀疏特征或增强视图之间的一致性。在概率模型中，最小化负对数似然等价于最大化赋给观测数据的概率；在判别式训练中，同一个标量也可以更直接地理解为改变参数的梯度信号。

目标函数是代理指标，而不是最终产品目标。cross-entropy 会奖励给标注类别分配经过校准的概率，但它并不直接编码公平性、延迟、拒答、排序效用，或某一类错误的成本。在改变优化器之前，应先验证标量目标是否确实表达了所关心的行为。训练损失持续下降，只能证明这个特定代理指标正在已观测样本上被优化。

归约方式本身也是目标设计的一部分。使用 `mean` 归约时，复制数据集中的每个样本不会改变参数梯度；使用 `sum` 归约时，梯度会乘以复制次数。因此，学习率、梯度累积和分布式数据并行中的平均行为，都必须结合损失归约方式来理解。

<details>
<summary><strong>PyTorch：观察经验风险归约与 L2 偏好</strong></summary>

~~~python
import copy
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(51)
x = torch.randn(6, 3)
targets = torch.tensor([0, 1, 2, 1, 0, 2])
template = nn.Linear(3, 3, bias=False)


def parameter_gradient(reduction: str, repeats: int) -> torch.Tensor:
    model = copy.deepcopy(template)
    logits = model(x.repeat((repeats, 1)))
    repeated_targets = targets.repeat(repeats)
    data_fit = F.cross_entropy(logits, repeated_targets, reduction=reduction)
    l2_preference = 0.05 * model.weight.square().sum()
    objective = data_fit + l2_preference
    objective.backward()
    return model.weight.grad.detach()


mean_once = parameter_gradient("mean", repeats=1)
mean_twice = parameter_gradient("mean", repeats=2)
sum_once = parameter_gradient("sum", repeats=1)
sum_twice = parameter_gradient("sum", repeats=2)
l2_gradient = 0.1 * template.weight.detach()

# The regularizer is unchanged, so compare the data term separately below.
assert torch.allclose(mean_once - l2_gradient, mean_twice - l2_gradient)
assert torch.allclose(sum_twice - l2_gradient, 2.0 * (sum_once - l2_gradient))

print("mean-reduced data-gradient norm:", float((mean_once - l2_gradient).norm()))
print("sum-reduced data-gradient ratio:", float((sum_twice - l2_gradient).norm() / (sum_once - l2_gradient).norm()))
~~~

</details>

示例中的 L2 梯度为 $2\lambda W$，因此即使数据梯度为零，它仍然存在。优化器不应暗中决定哪些行为更可取：损失定义目标，优化器只决定接近目标的路径。

**应用。** 这一框架适用于分类器、回归器、语言模型、检索系统、扩散模型和强化学习目标。对于每一类任务，都应先写清楚究竟是哪一个标量被平均、掩蔽、加权或正则化。

**对比总结。** 单样本损失规定何为错误；经验风险聚合已观测错误；正则化加入单个样本之外的偏好；优化器只对最终得到的梯度采取行动。

### **回归与分类损失** {#regression-classification-losses}

对于回归目标 $y\in\mathbb{R}$ 与预测 $\hat y$，常见损失函数对噪声和离群点做出不同假设：

$$
\ell_{\mathrm{MSE}}=(\hat y-y)^2,
\qquad
\ell_{\mathrm{MAE}}=|\hat y-y|,
$$

$$
\ell_{\mathrm{Huber},\delta}(r)=
\begin{cases}
\frac{1}{2}r^2, & |r|\leq\delta,\\
\delta\left(|r|-\frac{1}{2}\delta\right), & |r|>\delta,
\end{cases}
\qquad r=\hat y-y.
$$

MSE 是固定方差 Gaussian 观测噪声下的负对数似然目标。它的梯度随残差大小线性增长，因此较大的错误会获得不成比例的影响力。MAE 对离群点更稳健，但在零点不可微，并且在零点以外梯度幅值恒定。Huber 损失在拟合较好时为二次形式，在残差很大时为线性形式；当大多数标签可靠、但偶尔会出现较大误差时，它是很有用的折中方案。

对于分类，应在 **logits** 上训练，而不是在已经归一化的概率上训练。在二分类中，logit $z$ 变为 $p=\sigma(z)$，binary cross-entropy 为

$$
\ell_{\mathrm{BCE}}(z,y)
=-\left[y\log\sigma(z)+(1-y)\log(1-\sigma(z))\right].
$$

对于 $K$ 类，softmax cross-entropy 为

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},
\qquad
\ell_{\mathrm{CE}}(z,y)=-\log p_y.
$$

实际 API `binary_cross_entropy_with_logits` 或 `cross_entropy` 会通过稳定的 `logsumexp` 代数把归一化与对数损失融合起来。若先应用 `sigmoid` 或 `softmax` 再取对数，当模型自信地犯错时就可能下溢或上溢。有效导数仍然很简洁：对于 one-hot 目标 $q$，softmax cross-entropy 向 logits 传回 $p-q$。

<details>
<summary><strong>PyTorch：比较离群点梯度与稳定的 logit 损失</strong></summary>

~~~python
import torch
from torch.nn import functional as F

predictions = torch.tensor([0.2, -0.5, 5.0], requires_grad=True)
targets = torch.zeros(3)

mse = F.mse_loss(predictions, targets, reduction="none")
mae = F.l1_loss(predictions, targets, reduction="none")
huber = F.huber_loss(predictions, targets, delta=1.0, reduction="none")

mse.mean().backward(retain_graph=True)
mse_gradient = predictions.grad.detach().clone()
predictions.grad.zero_()
huber.mean().backward()
huber_gradient = predictions.grad.detach().clone()

# The outlier's MSE gradient grows with its residual; Huber's is capped here.
assert mse_gradient[-1].abs() > huber_gradient[-1].abs()
assert torch.allclose(huber_gradient[-1], torch.tensor(1.0 / 3.0))

extreme_logits = torch.tensor([-100.0, 100.0])
binary_targets = torch.tensor([1.0, 0.0])
stable_bce = F.binary_cross_entropy_with_logits(extreme_logits, binary_targets)
probabilities = extreme_logits.sigmoid()
naive_bce = -(binary_targets * probabilities.log() + (1 - binary_targets) * (1 - probabilities).log()).mean()

assert torch.isfinite(stable_bce)
assert not torch.isfinite(naive_bce)
print("per-example MSE:  ", mse.detach().tolist())
print("per-example Huber:", huber.detach().tolist())
print("stable BCE:", float(stable_bce))
~~~

</details>

**应用。** 当大偏差应受到强烈惩罚且 Gaussian 噪声假设合理时，选择 MSE；对稳健数值预测选择 Huber；对独立二元标签选择 BCE-with-logits；对每个样本只有一个互斥标签的任务选择 multiclass cross-entropy。多标签分类对每一个标签使用一个二元损失，而不是在标签之间使用 softmax cross-entropy。

**对比总结。** 损失函数是建模选择。MSE 强调大残差，MAE 限制大残差的影响，Huber 在两者间折中，而基于 logit 的 cross-entropy 将概率分类错误转化为稳定且有信息量的梯度。

### **序列、度量与类别不平衡损失** {#sequence-metric-imbalanced-losses}

许多深度学习任务并不是把一个输入映射为一个独立标签。语言模型或序列标注器会产生 logits $Z\in\mathbb{R}^{B\times T\times V}$：其中 $B$ 是序列数，$T$ 是位置数，$V$ 是词表或标签集合大小。token 负对数似然必须忽略 padding 位置，并且通常应按有效 token 数量归一化，而不是按 $B\times T$ 归一化：

$$
\mathcal{L}_{\mathrm{token}}
=-\frac{1}{\sum_{b,t}m_{bt}}
\sum_{b,t}m_{bt}\log p_\theta(y_{bt}\mid x_b,y_{b,<t}),
$$

其中 $m_{bt}\in\{0,1\}$ 是 attention 或 loss mask。若不做 masking，额外 padding 会改变目标函数，并让模型因预测 padding token 而获得奖励，而不是因预测有用内容而获益。

Label smoothing 会将 one-hot 目标 $q$ 替换为略微平滑的分布，从而减小模型必须对每个训练样本做出无限自信预测的压力。Focal loss 通常以 $(1-p_t)^\gamma\ell_{\mathrm{CE}}$ 下调简单样本的权重，而 class-weighted cross-entropy 会提高在欠代表类别上犯错的成本。这些工具针对的症状不同：smoothing 主要涉及置信度与正则化；focal loss 改变哪些样本主导梯度；class weights 则编码不对称的类别重要性。

重新加权也会改变总体目标。使用类别频率倒数作为权重训练的分类器，若不做适当修正，估计的不再是原始类别分布下的概率。它可能改善稀有类别的排序能力，却让原始概率失去良好校准。因此，应在未加权的验证分布上检查目标工作阈值、每类 precision/recall 与 calibration，而不能假设 weighted loss 更低就一定对应更好的部署决策。

度量学习目标定义的是样本之间的关系，而不只是类别标签。对于正样本对 $(a,p)$ 和负样本 $n$，triplet 风格目标要求正样本距离至少以一个 margin 小于负样本距离：

$$
\ell_{\mathrm{triplet}}
=\max\bigl(0, d(h_a,h_p)-d(h_a,h_n)+m\bigr).
$$

InfoNCE 等 contrastive 目标则让匹配表示的相似度高于 batch 中其他候选项。其效果高度依赖数据增强、负样本选择、temperature 与 batch 构造，因此不能脱离数据管道独立选择损失函数。

<details>
<summary><strong>PyTorch：计算带掩码 token 损失、label smoothing 与 focal weighting</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(53)
B, T, V = 2, 4, 5
logits = torch.randn(B, T, V)
targets = torch.tensor([[1, 3, 0, -100], [2, 4, -100, -100]])
valid_tokens = targets.ne(-100)

# PyTorch flattens token positions and ignores padding through ignore_index.
masked_ce = F.cross_entropy(
    logits.reshape(-1, V),
    targets.reshape(-1),
    ignore_index=-100,
    label_smoothing=0.1,
)

safe_targets = targets.masked_fill(~valid_tokens, 0)
per_token_ce = F.cross_entropy(logits.transpose(1, 2), safe_targets, reduction="none")
probability_of_target = logits.softmax(dim=-1).gather(-1, safe_targets.unsqueeze(-1)).squeeze(-1)
focal_weight = (1.0 - probability_of_target).square()
masked_focal = (per_token_ce * focal_weight * valid_tokens).sum() / valid_tokens.sum()

assert torch.isfinite(masked_ce)
assert torch.isfinite(masked_focal)
assert int(valid_tokens.sum()) == 5
print("smoothed masked token loss:", float(masked_ce))
print("focal-style masked loss:   ", float(masked_focal))
~~~

</details>

**应用。** 对 padding 后的语言、语音、视觉和图批次使用 mask；只有在数据分布与评估指标确实支持不对称处理时，才使用 weighted 或 focal 目标；对检索、验证、聚类和表示学习使用 metric loss。

**对比总结。** 序列损失决定哪些位置计入目标，类别不平衡损失决定哪些样本计入更多权重，度量损失决定样本之间应保留哪些关系。三者改变的都是梯度分布，而不只是最终标量数值。

### **梯度下降与随机梯度** {#gradient-descent-stochastic-gradients}

对于可微目标 $J(\theta)$，梯度下降应用

$$
\theta_{t+1}=\theta_t-\eta_t\nabla J(\theta_t),
$$

其中 $\eta_t$ 是学习率。Full-batch gradient descent 在每次更新前计算全部 $N$ 个样本的平均梯度。它在概念上清晰，但对大型数据集开销高，并且每完整遍历一次数据只能进行一次更新。

学习率必须结合曲率来解释。对于正定 Hessian 为 $H$ 的二次目标 $J(\theta)=\frac{1}{2}\theta^\top H\theta$，固定步长梯度下降仅在下列范围内稳定：

$$
0<\eta<\frac{2}{\lambda_{\max}(H)}.
$$

沿特征值为 $\lambda_i$ 的特征向量方向，每一步误差都会乘以 $1-\eta\lambda_i$。学习率超过上界时，至少有一个方向会发散；学习率很小时虽稳定，却收敛缓慢。较大的条件数 $\kappa=\lambda_{\max}/\lambda_{\min}$ 会产生熟悉的锯齿路径：学习率必须服从陡峭方向的限制，而沿平缓方向的进展仍然很小。Momentum 与自适应预条件化在一定程度上正是为了改善这种几何结构。

Stochastic gradient descent（SGD）改用一个样本或一个小 minibatch $\mathcal{B}_t$：

$$
g_t=
\frac{1}{|\mathcal{B}_t|}\sum_{i\in\mathcal{B}_t}
\nabla_\theta\ell_i(\theta_t),
\qquad
\theta_{t+1}=\theta_t-\eta_tg_t.
$$

当 batch 以合适方式采样时，$g_t$ 是完整经验梯度的无偏估计，但它具有方差。这种噪声不仅仅是实现缺陷：它使频繁且成本较低的更新成为可能，能够帮助优化跨越浅层势垒，并与泛化相互作用。同时，它也意味着单个带噪损失值或一次更新方向，都是训练是否健康的很弱证据。

Minibatch 是实践中的折中。它们降低梯度估计器的方差，能高效利用矩阵硬件，同时仍比 full-batch 优化更频繁地更新。增大 batch size 同时改变统计估计和系统行为，不能把它当成无害的速度设置。

<details>
<summary><strong>PyTorch：测量随机梯度方差，并使用 minibatch 训练</strong></summary>

~~~python
import torch

torch.manual_seed(57)
N = 256
features = torch.randn(N)
targets = 2.5 * features


def batch_gradient(weight: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
    prediction = weight * features[indices]
    return (2.0 * features[indices] * (prediction - targets[indices])).mean()


weight_at_start = torch.tensor(0.0)
generator = torch.Generator().manual_seed(58)
single_estimates = torch.stack([
    batch_gradient(weight_at_start, torch.randint(N, (1,), generator=generator))
    for _ in range(400)
])
minibatch_estimates = torch.stack([
    batch_gradient(weight_at_start, torch.randint(N, (32,), generator=generator))
    for _ in range(400)
])


def train_with_batch_size(batch_size: int) -> torch.Tensor:
    weight = torch.tensor(0.0)
    local_generator = torch.Generator().manual_seed(59 + batch_size)
    for _ in range(80):
        indices = torch.randint(N, (batch_size,), generator=local_generator)
        weight -= 0.1 * batch_gradient(weight, indices)
    return weight


single_solution = train_with_batch_size(1)
minibatch_solution = train_with_batch_size(32)
assert minibatch_estimates.var() < single_estimates.var()
assert abs(minibatch_solution - 2.5) < 0.1
print("single-example gradient variance:", float(single_estimates.var()))
print("minibatch gradient variance:     ", float(minibatch_estimates.var()))
print("learned weights (SGD, minibatch):", float(single_solution), float(minibatch_solution))
~~~

</details>

**应用。** 几乎所有大型神经网络都使用 minibatch stochastic gradient。相同原则也出现在在线学习、流式适应与联邦优化中，只是它们的采样假设和噪声来源有所不同。

**对比总结。** Full-batch descent 使用精确的数据集梯度，但更新很少；SGD 使用高方差、成本低的估计；minibatch SGD 则用部分随机性换取更稳定且更适合硬件的更新。

### **动量与 Nesterov 加速** {#momentum-nesterov-acceleration}

普通 SGD 只对当前 minibatch 梯度做出反应。在狭窄且弯曲的谷地中，它可能沿陡峭方向剧烈振荡，却沿平缓方向前进缓慢。**Momentum** 保留一个类似速度的梯度指数移动平均：

$$
v_t=\beta v_{t-1}+g_t,
\qquad
\theta_{t+1}=\theta_t-\eta v_t,
$$

其中 $\beta\in[0,1)$ 控制记忆长度。使用该约定时，近期且一致的梯度会逐渐积累速度，而快速变化、有噪声的方向会部分相互抵消。有些库会把新梯度乘以 $(1-\beta)$；这会改变 $v_t$ 的数值尺度，但不会改变更新保存递减历史这一核心思想。

**Nesterov accelerated gradient（NAG）**在预期位置而非当前位置计算梯度。一种常见形式是

$$
g_t=\nabla J(\theta_t-\eta\beta v_{t-1}),
\qquad
v_t=\beta v_{t-1}+g_t,
\qquad
\theta_{t+1}=\theta_t-\eta v_t.
$$

这个 lookahead 本质上在询问：当前速度是否将要越过合适区域。它可以比普通动量更早开始修正，但仍需要合适的学习率，也不能消除所有不稳定性。

![Momentum 会在时间上聚合梯度，比未加速更新更直接地到达条件不良二次函数的最小值。](assets/dl05-momentum-trajectory.svg){fig-align="center" width="72%" fig-alt="一个二维二次优化问题的等高线图，其中 momentum 轨迹逐步逼近最小值。"}

*图片来源：[Dive into Deep Learning, Momentum](https://d2l.ai/chapter_optimization/momentum.html)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

<details>
<summary><strong>PyTorch：在条件不良二次函数上比较 SGD、momentum 和 Nesterov</strong></summary>

~~~python
import torch

curvature = torch.tensor([1.0, 30.0])


def quadratic(theta: torch.Tensor) -> torch.Tensor:
    return 0.5 * (curvature * theta.square()).sum()


def optimize(method: str, steps: int = 80, lr: float = 0.02, beta: float = 0.9):
    theta = torch.tensor([4.0, 4.0])
    velocity = torch.zeros_like(theta)
    losses = []
    for _ in range(steps):
        if method == "nesterov":
            gradient = curvature * (theta - lr * beta * velocity)
        else:
            gradient = curvature * theta

        if method == "sgd":
            theta = theta - lr * gradient
        else:
            velocity = beta * velocity + gradient
            theta = theta - lr * velocity
        losses.append(quadratic(theta))
    return torch.stack(losses)


sgd_losses = optimize("sgd")
momentum_losses = optimize("momentum")
nesterov_losses = optimize("nesterov")

assert torch.isfinite(torch.stack((sgd_losses[-1], momentum_losses[-1], nesterov_losses[-1]))).all()
assert momentum_losses[-1] < sgd_losses[-1]
assert nesterov_losses[-1] < sgd_losses[-1]
print("final losses:", {"sgd": float(sgd_losses[-1]), "momentum": float(momentum_losses[-1]), "nesterov": float(nesterov_losses[-1])})
~~~

</details>

**应用。** 在精心调优后，momentum SGD 仍是视觉训练中常见且强大的 baseline。当实现直接支持 Nesterov momentum 时，它也很有用，但它的增益取决于曲率、batch 噪声、调度策略和其他选择。

**对比总结。** SGD 只跟随当前梯度；momentum 通过速度状态过滤梯度；Nesterov momentum 计算 lookahead gradient。两种加速方法都以额外状态和超参数，换取沿持续方向更快的移动。

### **自适应优化器** {#adaptive-optimizers}

优化器不仅可以在时间上平均梯度，还可以对不同坐标进行重缩放。当不同参数收到的梯度幅值相差很大时，这一点特别有用。**AdaGrad** 为每个坐标累积一个非负二阶矩估计：

$$
s_t=s_{t-1}+g_t\odot g_t,
\qquad
\theta_{t+1}=\theta_t-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}.
$$

频繁活跃的坐标会获得逐渐变小的步长。这对稀疏特征可能很有价值，但分母只增不减，因此学习最终可能变得过慢。

**RMSProp** 使用指数移动平均替代无界累积和：

$$
s_t=\rho s_{t-1}+(1-\rho)g_t\odot g_t,
\qquad
\theta_{t+1}=\theta_t-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}.
$$

该分母相当于一个对角预条件器：持续出现的大幅、带噪坐标会比小坐标受到更强抑制。它不是完整的曲率矩阵，也不保证带来更好的泛化。自适应重缩放改变了优化几何，这正是学习率和 weight decay 设置不应盲目从 SGD 复制过来的原因。

<details>
<summary><strong>PyTorch：观察 AdaGrad 与 RMSProp 的逐坐标缩放</strong></summary>

~~~python
import torch

gradient = torch.tensor([100.0, 1.0])
learning_rate = 0.1
epsilon = 1e-8

sgd_update = learning_rate * gradient
adagrad_state = gradient.square()
adagrad_update = learning_rate * gradient / (adagrad_state.sqrt() + epsilon)

rho = 0.9
rmsprop_state = (1.0 - rho) * gradient.square()
rmsprop_update = learning_rate * gradient / (rmsprop_state.sqrt() + epsilon)

assert sgd_update[0] / sgd_update[1] == 100.0
assert torch.allclose(adagrad_update, torch.tensor([0.1, 0.1]), atol=1e-6)
assert torch.allclose(rmsprop_update[0], rmsprop_update[1], atol=1e-6)
print("SGD update:    ", sgd_update.tolist())
print("AdaGrad update:", adagrad_update.tolist())
print("RMSProp update:", rmsprop_update.tolist())
~~~

</details>

**应用。** 自适应方法常被用作 transformer、语言模型、稀疏问题和快速原型的便捷默认选择。当梯度尺度在坐标之间差异很大时，它们尤其有帮助；但获得好结果仍需配合调度、合理正则化，并使用任务指标验证。

**对比总结。** Momentum 平均梯度的一阶矩；AdaGrad 累积全部平方梯度；RMSProp 对平方梯度做指数平均。这些方法解决的是不同的条件数与噪声问题，也可以像 Adam 一样被组合使用。

### **AdamW 与解耦的 Weight Decay** {#adamw-decoupled-weight-decay}

Adam 将一阶矩 momentum 与 RMSProp 风格的二阶矩结合起来。给定梯度 $g_t$，它形成

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,
\qquad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2,
$$

然后对从零开始的估计进行 bias correction：

$$
\hat m_t=\frac{m_t}{1-\beta_1^t},
\qquad
\hat v_t=\frac{v_t}{1-\beta_2^t},
\qquad
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$

通常的默认值 $\beta_1=0.9$、$\beta_2=0.999$ 和 $\epsilon=10^{-8}$ 只是起点，并不是定律。由于每个坐标都有自己的分母，把 L2 penalty 梯度 $\lambda\theta$ 加入 Adam 通常并不等价于按固定衰减因子缩小参数；自适应分母也会重缩放正则化信号。

**AdamW** 将这两个操作解耦：

$$
\theta'_{t}= (1-\eta\lambda)\theta_t,
\qquad
\theta_{t+1}=\theta'_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$

不同实现的操作顺序可能略有差异，但关键性质是：weight decay 直接作用于参数值，而不是进入 Adam 的自适应 moment estimate。除非有明确理由，否则应将 bias 向量和 normalization 的 scale/shift 排除在 decay 之外。

优化器状态也是一项内存决策。对于以 FP32 存储的 $P$ 个可训练标量参数，下列粗略估算不包含 activation、临时缓冲区、内存分配器开销和分布式副本：

| 优化器 | 每个参数的持久状态 | 参数 + 梯度 + 优化器状态的近似大小 |
|---|---:|---:|
| SGD | 无 | $8P$ bytes |
| Momentum SGD | 一个 velocity | $12P$ bytes |
| AdaGrad / RMSProp | 一个平方梯度累加器 | $12P$ bytes |
| Adam / AdamW | 一阶矩和二阶矩 | $16P$ bytes |

带 momentum 的 RMSProp 还需要一个 velocity tensor。一些混合精度实现还会为低精度参数保留 FP32 master copy，额外增加约 $4P$ bytes。这解释了为什么即使前向架构完全不变，从 SGD 改成 AdamW 也可能明显缩小能够放入显存的模型或 batch size。

<details>
<summary><strong>PyTorch：在零数据梯度下观察耦合 L2 正则与 AdamW decay 的区别</strong></summary>

~~~python
import torch

learning_rate = 0.1
weight_decay = 0.1
adam_parameter = torch.nn.Parameter(torch.tensor([1.0]))
adamw_parameter = torch.nn.Parameter(torch.tensor([1.0]))

adam = torch.optim.Adam([adam_parameter], lr=learning_rate, weight_decay=weight_decay)
adamw = torch.optim.AdamW([adamw_parameter], lr=learning_rate, weight_decay=weight_decay)

# There is no data gradient. Any movement comes from regularization/decay.
adam_parameter.grad = torch.zeros_like(adam_parameter)
adamw_parameter.grad = torch.zeros_like(adamw_parameter)
adam.step()
adamw.step()

assert torch.allclose(adamw_parameter.detach(), torch.tensor([0.99]), atol=1e-6)
assert adam_parameter.detach() < adamw_parameter.detach()
print("Adam with coupled L2: ", float(adam_parameter.detach()))
print("AdamW decay:          ", float(adamw_parameter.detach()))
~~~

</details>

**应用。** AdamW 是 transformer 风格模型与许多现代预训练网络的常用优化器 baseline。实际配置是一组相互联系的选择：学习率、weight decay、parameter groups、warmup、总更新数、gradient clipping、精度和有效 batch size 都会相互作用。

**对比总结。** Adam 使用两个 moment estimate 自适应方向和尺度；AdamW 在保留该自适应更新的同时，对选定 parameter groups 施加独立且可预测的收缩。

### **学习率调度与 Warmup** {#learning-rate-schedules-warmup}

学习率控制梯度转化为参数变化的尺度。一个在训练早期有用的固定学习率，靠近解时可能过大并导致持续振荡；一个适合后期精细收敛的学习率，又会让早期进展过慢。因此，**learning-rate schedule** 会让 $\eta_t$ 成为更新次数、epoch、验证表现或训练阶段的有意函数。

常见调度包括 step decay、linear decay、cosine decay，以及由验证集触发的降低。Cosine decay 在 $T$ 次更新内把学习率从 $\eta_{max}$ 平滑降低到 $\eta_{min}$：

$$
\eta_t=\eta_{min}+\frac{1}{2}(\eta_{max}-\eta_{min})
\left(1+\cos\left(\pi\frac{t}{T}\right)\right).
$$

**Warmup** 从较小的学习率开始，并在前 $W$ 次更新内逐步提高，通常是线性提高。当随机初始化、大有效 batch、自适应优化器状态、normalization statistics 或预训练模型适应使前几次大更新不可靠时，它尤其有价值。一个简单的 warmup 后接 cosine decay 的形式为

$$
\eta_t=
\begin{cases}
\eta_{max}\frac{t+1}{W}, & t<W,\\
\eta_{min}+\frac{1}{2}(\eta_{max}-\eta_{min})
\left[1+\cos\left(\pi\frac{t-W+1}{T-W}\right)\right], & t\geq W.
\end{cases}
$$

计量单位很重要。若调度按 *optimizer updates* 定义，当梯度累积、数据并行 world size、数据集大小或 batch size 改变时，调度也会改变。应明确记录总更新数与 warmup 更新数，而不是只用 epoch 描述调度。

![Warmup 会逐渐升高学习率，随后 cosine schedule 使学习率降低，以便在训练后期进行精细优化。](assets/dl05-warmup-cosine-schedule.svg){fig-align="center" width="72%" fig-alt="一张折线图，展示短暂的线性学习率 warmup，之后是平缓的 cosine 形下降。"}

*图片来源：[Dive into Deep Learning, Learning Rate Scheduling](https://d2l.ai/chapter_optimization/lr-scheduler.html)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

<details>
<summary><strong>PyTorch：构造并检查 warmup 加 cosine 的更新调度</strong></summary>

~~~python
import math
import torch


def warmup_cosine_lr(step: int, total_updates: int, warmup_updates: int, base_lr: float, final_lr: float) -> float:
    if not 0 <= step < total_updates:
        raise ValueError("step must be inside the training schedule")
    if step < warmup_updates:
        return base_lr * (step + 1) / warmup_updates
    progress = (step - warmup_updates + 1) / (total_updates - warmup_updates)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return final_lr + (base_lr - final_lr) * cosine


total_updates, warmup_updates = 20, 4
base_lr, final_lr = 3e-3, 1e-4
learning_rates = [warmup_cosine_lr(step, total_updates, warmup_updates, base_lr, final_lr) for step in range(total_updates)]

# LambdaLR can use the same update-indexed function in a real optimizer.
parameter = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([parameter], lr=base_lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: warmup_cosine_lr(step, total_updates, warmup_updates, base_lr, final_lr) / base_lr,
)

# Exercise the scheduler at optimizer-update granularity, not merely construct it.
observed_rates = [optimizer.param_groups[0]["lr"]]
for _ in range(1, total_updates):
    optimizer.zero_grad(set_to_none=True)
    parameter.grad = torch.zeros_like(parameter)
    optimizer.step()
    scheduler.step()
    observed_rates.append(optimizer.param_groups[0]["lr"])

assert learning_rates[0] < learning_rates[warmup_updates - 1] == base_lr
assert learning_rates[-1] == final_lr
assert all(left >= right for left, right in zip(learning_rates[warmup_updates - 1 :], learning_rates[warmup_updates:]))
assert torch.allclose(torch.tensor(observed_rates), torch.tensor(learning_rates), atol=1e-9)
print("first six learning rates:", [round(rate, 6) for rate in learning_rates[:6]])
print("last learning rate:      ", learning_rates[-1])
~~~

</details>

**应用。** Warmup 加 cosine decay 是许多 transformer 与视觉训练的强大显式 baseline。Step decay 对成熟 recipe 仍然有效，而 plateau scheduling 在训练进度由验证反馈而不是固定更新预算决定时更有用。

**对比总结。** 优化器决定梯度如何被预条件化；调度决定该变换后梯度在每个阶段推动参数多远。Warmup 保护早期动态，decay 支持后期精细优化。

### **Batch Size、梯度累积与梯度裁剪** {#batch-size-accumulation-clipping-mixed-precision}

**有效 batch size（effective batch size）**是一次优化器更新中贡献梯度的样本数量。若每个设备使用 microbatch $b$，梯度累积 $a$ 次，数据并行使用 $w$ 个 worker，则

$$
B_{\mathrm{effective}}=b\times a\times w.
$$

当一个 microbatch 已经占满显存时，梯度累积可以实现更大的有效 batch。对于 mean-reduced loss，应在 `.backward()` 前将每个 microbatch loss 除以 $a$，使累积梯度与拼接后完整 batch 的梯度一致。只有在有效 batch 完成后，才执行 optimizer 和 scheduler step；在 microbatch 之间调用 `zero_grad()` 会丢掉部分梯度和。

增大有效 batch 通常会降低梯度噪声，但也会减少每个 epoch 的更新次数。常见的线性缩放启发式，即 batch size 增大 $k$ 倍时令 $\eta'\approx k\eta$，只是依赖具体 recipe 的起点，并不是定律；曲率、优化器、normalization、数据冗余和训练更新数都可能使它失效。大 batch 训练常把重新调优的学习率与 warmup 配合使用，但只有同时说明更新预算和数据暴露量，这种比较才有意义。

**Gradient clipping** 会限制异常大的更新。全局范数裁剪将梯度向量 $g$ 替换为

$$
g\leftarrow g\min\left(1,\frac{c}{\lVert g\rVert_2+\epsilon}\right),
$$

其中 $c$ 是 `max_norm`。它尤其适用于循环、生成式和早期不稳定训练。裁剪是一种安全机制，而不是有效损失、稳定初始化或合适学习率的替代品。若它几乎每一步都被触发，应诊断根因，而不是仅仅降低阈值。

**Automatic mixed precision（AMP）**使用较低精度执行选定操作，在保留敏感计算较安全精度的同时降低内存并提高加速器吞吐。使用 FP16 时，小梯度可能下溢为零。`torch.amp.GradScaler` 会在反向传播前放大损失，并在 optimizer step 前取消放大；它还可以跳过包含非有限梯度的更新。关键点是：检查或裁剪梯度之前必须先 unscale。BF16 与 FP32 有相同的 exponent range，通常不需要 gradient scaling，不过它仍会改变精度。

对于使用 FP16 的梯度累积，操作顺序应为：每个有效 batch 只执行一次 `zero_grad` $\rightarrow$ 在 autocast 中执行 forward 与 loss $\rightarrow$ 对每个 microbatch 缩放 loss 并调用 `backward` $\rightarrow$ 只 unscale 一次 $\rightarrow$ 检查或裁剪梯度 $\rightarrow$ `scaler.step` $\rightarrow$ `scaler.update` $\rightarrow$ 推进按 update 计数的 scheduler。在同一份累积梯度仍未形成完毕时，不应反复更新 scale 或重复 unscale。

![不同神经网络工作负载中，AMP 相对 FP32 的实测加速差异明显。](assets/dl05-pytorch-amp-speedup.png){fig-align="center" width="72%" fig-alt="一张柱状图，对比 BERT、GNMT、NCF、ResNet、SSD、Tacotron、Transformer XL 和 WaveGlow 工作负载中混合精度相对 FP32 的加速比。"}

*图片来源：[PyTorch, What Every User Should Know About Mixed Precision Training in PyTorch](https://pytorch.org/blog/what-every-user-should-know-about-mixed-precision-training-in-pytorch/)。该 benchmark 说明 AMP 是需要实测的系统优化，而不是固定加速倍数的保证。*

Distributed data parallelism 会贡献另一份经平均的梯度，因此会改变有效 batch size。其系统实现将在第 19 章的可扩展训练中展开；在这里，应把 world size 视为更新定义的一部分，而不是仅仅视为部署细节。

<details>
<summary><strong>PyTorch：让梯度累积匹配大 batch，然后使用 AMP 安全的裁剪顺序</strong></summary>

~~~python
import copy
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(61)
features = torch.randn(8, 3)
targets = torch.randn(8, 2)
template = nn.Linear(3, 2)
large_batch_model = copy.deepcopy(template)
accumulated_model = copy.deepcopy(template)

large_optimizer = torch.optim.SGD(large_batch_model.parameters(), lr=0.05)
accumulated_optimizer = torch.optim.SGD(accumulated_model.parameters(), lr=0.05)

# One full batch and two microbatches should produce the same clipped update.
large_optimizer.zero_grad(set_to_none=True)
large_loss = F.mse_loss(large_batch_model(features), targets)
large_loss.backward()
torch.nn.utils.clip_grad_norm_(large_batch_model.parameters(), max_norm=1.0)
large_optimizer.step()

accumulated_optimizer.zero_grad(set_to_none=True)
for micro_features, micro_targets in zip(features.chunk(2), targets.chunk(2)):
    micro_loss = F.mse_loss(accumulated_model(micro_features), micro_targets)
    (micro_loss / 2).backward()  # Divide by accumulation steps for mean-reduced loss.
torch.nn.utils.clip_grad_norm_(accumulated_model.parameters(), max_norm=1.0)
accumulated_optimizer.step()

assert all(torch.allclose(a, b, atol=1e-6) for a, b in zip(large_batch_model.parameters(), accumulated_model.parameters()))

# The same ordering is safe with AMP. It runs as ordinary FP32 when CUDA is unavailable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_model = nn.Linear(3, 2).to(device)
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)
amp_optimizer.zero_grad(set_to_none=True)

with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
    amp_loss = F.mse_loss(amp_model(features.to(device)), targets.to(device))
scaler.scale(amp_loss).backward()
scaler.unscale_(amp_optimizer)  # Gradients must be unscaled before clipping.
unscaled_norm = torch.nn.utils.clip_grad_norm_(amp_model.parameters(), max_norm=1.0)
scaler.step(amp_optimizer)
scaler.update()

assert torch.isfinite(torch.stack([parameter.detach().abs().max().cpu() for parameter in amp_model.parameters()])).all()
print("accumulation matches full batch:", True)
print("pre-clipping AMP gradient norm:", float(unscaled_norm))
~~~

</details>

**应用。** 梯度累积常见于显存受限的语言、视觉和多模态训练。AMP 是现代加速器上的标准性能技术。二者都要求仔细记账：loss reduction、缩放因子、裁剪位置、scheduler step 与 optimizer step 必须共同对应同一次有效更新。

**对比总结。** Batch size 控制一次估计背后的数据量；累积在时间上模拟更大的 batch；裁剪为异常梯度设定上界；AMP 改变算术精度，因此需要正确的缩放与取消缩放顺序。

### **初始化与早期训练动态** {#initialization-early-training-dynamics}

初始化决定模型最初的激活、梯度和对称性。如果两个隐藏单元以相同参数开始，并接收相同输入，它们会在梯度下降中始终相同，因而无法学习不同特征。随机初始化会打破这种对称性。其方差还必须与架构相匹配：若数值在多层间反复收缩或增长，就会在优化尚未来得及帮助之前产生接近零或爆炸的激活与梯度。

对于 fan-in 为 $n_{in}$、fan-out 为 $n_{out}$ 的层，Xavier/Glorot 初始化试图为大致线性或对称激活保持方差，常使用接近下式的方差：

$$
\operatorname{Var}(W)\approx\frac{2}{n_{in}+n_{out}}.
$$

对于 ReLU 类激活，He/Kaiming 初始化会补偿大约一半激活被移除的效果：

$$
\operatorname{Var}(W)\approx\frac{2}{n_{in}}.
$$

这些是方差保持启发式，而不是普适保证。残差路径、normalization、embedding scale、深度、优化器和精度都会改变早期动态。有用的做法是将合理默认值与测量结合：在最初几百次更新中检查 activation scale、gradient norms、更新与参数的比值以及非有限数值。

<details>
<summary><strong>PyTorch：对比保留对称性的零初始化与 Kaiming 初始化</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class SmallReLUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(6, 12)
        self.output = nn.Linear(12, 1)

    def forward(self, x):
        hidden = F.relu(self.hidden(x))
        return self.output(hidden), hidden


def initialize(model: nn.Module, mode: str) -> None:
    if mode == "zero":
        nn.init.zeros_(model.hidden.weight)
        nn.init.zeros_(model.output.weight)
    elif mode == "kaiming":
        # Match Kaiming to the hidden ReLU, but use Xavier for the linear output.
        nn.init.kaiming_normal_(model.hidden.weight, nonlinearity="relu")
        nn.init.xavier_normal_(model.output.weight)
    else:
        raise ValueError(mode)
    nn.init.zeros_(model.hidden.bias)
    nn.init.zeros_(model.output.bias)


torch.manual_seed(67)
inputs = torch.randn(32, 6)
targets = torch.randn(32, 1)
zero_model, kaiming_model = SmallReLUNet(), SmallReLUNet()
initialize(zero_model, "zero")
initialize(kaiming_model, "kaiming")

zero_prediction, zero_hidden = zero_model(inputs)
zero_loss = F.mse_loss(zero_prediction, targets)
zero_loss.backward()
kaiming_prediction, kaiming_hidden = kaiming_model(inputs)
kaiming_loss = F.mse_loss(kaiming_prediction, targets)
kaiming_loss.backward()

assert zero_hidden.var() == 0
assert zero_model.hidden.weight.grad.abs().sum() == 0
assert kaiming_hidden.var() > 0
assert kaiming_model.hidden.weight.grad.abs().sum() > 0
print("zero-init hidden variance:   ", float(zero_hidden.detach().var()))
print("Kaiming hidden variance:     ", float(kaiming_hidden.detach().var()))
print("Kaiming first-layer grad norm:", float(kaiming_model.hidden.weight.grad.norm()))
~~~

</details>

**应用。** 从框架默认值或架构推荐初始化开始，然后在调优复杂优化器之前检查早期动态。增加新的残差分支、自定义 normalization、非常规激活，或很深的循环/状态空间组件时，这一点尤为重要。

**对比总结。** 初始化提供起始分布，优化提供迭代移动。较差初始化会从第一步起就让梯度失去信息，而尺度合适的随机初始化会打破对称性，并为优化提供可用信号。

### **损失曲面与优化诊断** {#loss-landscapes-optimization-diagnostics}

**损失曲面（loss landscape）**指的是目标值作为全部参数函数的形状。现代网络的这一空间拥有数百万甚至数十亿维，因此二维等高线图只能是局部投影。即便如此，这类图仍提供有用直觉：狭窄谷地会产生振荡，平坦或条件不良方向会让进展缓慢，鞍点附近可能梯度很小却并非好解，而不同参数化可能在不同位置表达相同函数。

不要只根据损失诊断真实训练。一个有用的训练记录至少应包含：

| 信号 | 它可能揭示什么 | 注意事项 |
|---|---|---|
| train loss 与任务指标 | 目标进展和面向任务的行为 | 低训练损失可与较差验证表现共存 |
| validation loss 与指标 | 泛化和过拟合 | 应在稳定的评估协议下比较 |
| learning rate 与更新次数 | 预期调度是否真的生效 | 记录实际 optimizer 值，而不只记录配置 |
| 全局或逐层 gradient norms | 爆炸、消失或断开的路径 | 范数必须结合尺度与层类型解释 |
| parameter norm 与更新/参数比值 | 更新是否微弱或具有破坏性 | 应跨时间比较，而不是只看单步 |
| activation distributions 与非有限值计数 | 饱和、死亡单元、溢出或无效数据 | 检查代表性层，而非每个张量 |

不同症状对应不同干预。损失立即变为 `NaN` 可能表示无效数据、数值操作、AMP 溢出或过大的更新；持续接近零的梯度可能表示饱和、masking、断开的计算图或尺度较差；训练改进而验证变差则意味着过拟合，或目标与评估不匹配。降低学习率可能有用，但不是通用诊断方法。

最可靠的调试流程是在一个小型、确定性的运行中一次只改变一个有意义变量。先在极小子集上过拟合，然后确认目标函数、数据管道、更新记账、precision mode 与指标。只有在这些步骤完成后，大规模超参数搜索才值得信任。

<details>
<summary><strong>PyTorch：在小型训练中记录 loss、gradient norm 与相对更新大小</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(71)
features = torch.randn(128, 4)
true_weight = torch.tensor([1.2, -0.8, 0.5, 1.0])
labels = ((features @ true_weight) > 0).float().unsqueeze(1)

model = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-2, weight_decay=1e-3)
ledger = []


def parameter_vector(module: nn.Module) -> torch.Tensor:
    return torch.cat([parameter.detach().flatten() for parameter in module.parameters()])


for step in range(40):
    before = parameter_vector(model)
    optimizer.zero_grad(set_to_none=True)
    loss = F.binary_cross_entropy_with_logits(model(features), labels)
    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    after = parameter_vector(model)
    update_ratio = (after - before).norm() / before.norm().clamp_min(1e-12)
    ledger.append(
        {
            "step": step,
            "loss": float(loss.detach()),
            "gradient_norm": float(gradient_norm),
            "update_ratio": float(update_ratio),
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

assert ledger[-1]["loss"] < ledger[0]["loss"]
assert all(torch.isfinite(torch.tensor(row["gradient_norm"])) for row in ledger)
print("first step:", ledger[0])
print("last step: ", ledger[-1])
~~~

</details>

**应用。** 这些诊断属于每一个严肃训练循环，无论是笔记本电脑实验还是分布式预训练运行。具体的日志系统可以变化，但将优化过程视为时间序列来观察的习惯具有普适性。

**对比总结。** 损失曲面提供有用直觉，诊断记录提供可操作证据。损失值展示所选标量是否降低；梯度、更新、激活、精度与验证信号则解释一次运行为什么稳定、停滞或具有误导性。

### **本章对比与总结** {#chapter-comparison-summary}

训练是目标函数、随机梯度估计、更新规则和测量过程的反复组合。不存在一种脱离数据、架构、batch 方案、精度和评估目标而普适最好的优化器或损失函数。

| 组件 | 核心问题 | 典型机制 | 重要失败模式 |
|---|---|---|---|
| 目标函数 | 什么行为会得到奖励？ | 经验风险加显式偏好 | 优化遗漏任务目标的代理指标 |
| 回归或分类损失 | 如何衡量预测错误？ | MSE、Huber、BCE-with-logits、cross-entropy | 不稳定的概率计算或不匹配的标签 |
| 序列或度量损失 | 哪些位置、样本或关系重要？ | masks、weights、smoothing、contrastive terms | padding、类别不平衡或负样本选择错误 |
| Minibatch SGD | 如何估计数据集梯度？ | 随机 batch 平均 | 方差过高或归约尺度错误 |
| Momentum/Nesterov | 如何加速持续方向？ | velocity 与 lookahead | 学习率与状态不兼容造成过冲 |
| 自适应方法 | 如何重缩放不同坐标？ | 二阶矩对角预条件 | 不经重新调优就复制 SGD 超参数 |
| AdamW | 如何分离自适应更新与收缩？ | Adam moments 加解耦 decay | 意外衰减不应衰减的参数，或误用耦合 L2 |
| 调度或 warmup | 步长如何随时间变化？ | 按更新索引的学习率 | 更新次数变化时仍按 epoch 调度 |
| 累积、AMP、裁剪 | 如何安全执行一次有效更新？ | 缩放的 microbatch 梯度、unscale、clip、step | 在错误粒度上 step、clip 或调度 |
| 初始化或诊断 | 早期训练信号是否可用？ | 方差感知初始化与记录统计量 | 把症状误认为根因 |

本章的主要结论是：

1. 损失函数编码的是建模与产品决策，而不是中性的实现细节。
2. 损失归约、batch size、累积次数和数据并行 world size 共同定义梯度尺度。
3. 稳定的基于 logit 的损失可避免数值失败，并为分类提供直接梯度。
4. Masks、class weights、focal factors 与 metric terms 会重新分配位置、样本和关系之间的梯度影响。
5. Minibatch gradients 是带噪估计；它们的噪声、更新频率和硬件效率都是训练设计的一部分。
6. Momentum 平均梯度方向，而自适应优化器借助梯度历史重缩放坐标。
7. AdamW 将 weight decay 与自适应梯度归一化解耦，并应使用有意设计的 parameter groups。
8. Learning-rate schedules 与 warmup 是按更新次数计的控制策略，而不是可选的装饰性设置。
9. Accumulation、clipping、AMP 与分布式平均必须围绕同一次有效更新按正确顺序与尺度执行。
10. 初始化决定在优化器发挥作用前，是否已经存在有用的打破对称的激活与梯度。
11. 诊断应追踪损失、验证、梯度、更新、激活、精度和非有限值构成的时间序列。

第 06 章研究模型拟合训练目标之后会发生什么：泛化、正则化、实验设计，以及判断一次训练改进是真实的还是某个 split 或 seed 的偶然产物所需的证据。